In [2]:
import cv2
import easyocr
import numpy as np
from tkinter import *
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk

# OCR engine
reader = easyocr.Reader(['en'], gpu=False)

# Fungsi untuk menghitung posisi tengah Y dari bounding box
def y_center(bbox):
    return np.mean([pt[1] for pt in bbox])

# Fungsi untuk memproses gambar
def proses_gambar():
    file_path = filedialog.askopenfilename(filetypes=[("Image Files", "*.jpg *.jpeg *.png")])
    if not file_path:
        return

    image = cv2.imread(file_path)
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hasil_ocr = reader.readtext(img_rgb)

    if not hasil_ocr:
        hasil_teks.delete('1.0', END)
        hasil_teks.insert(END, "Tidak ada teks terdeteksi.")
        return

    baris = []
    threshold = 25

    for bbox, teks, _ in hasil_ocr:
        y = y_center(bbox)
        masuk = False
        for b in baris:
            if abs(y - b[0][2]) < threshold:
                b.append((bbox, teks, y))
                masuk = True
                break
        if not masuk:
            baris.append([(bbox, teks, y)])

    hasil_atas, hasil_bawah = "", ""
    baris.sort(key=lambda b: min(item[2] for item in b))
    if len(baris) > 0:
        baris[0].sort(key=lambda x: x[0][0][0])
        hasil_atas = " ".join([text for _, text, _ in baris[0]])
    if len(baris) > 1:
        baris[1].sort(key=lambda x: x[0][0][0])
        hasil_bawah = " ".join([text for _, text, _ in baris[1]])

    # Gambar bounding box dan teks
    for bbox, teks, _ in hasil_ocr:
        p1 = tuple(map(int, bbox[0]))
        p2 = tuple(map(int, bbox[2]))
        cv2.rectangle(img_rgb, p1, p2, (0, 255, 0), 2)
        cv2.putText(img_rgb, teks, p1, cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    # Konversi ke PIL dan tampilkan di canvas
    img_tampil = Image.fromarray(img_rgb).resize((250, 150))  # Ukuran lebih kecil
    img_tk = ImageTk.PhotoImage(img_tampil)

    canvas.image = img_tk
    canvas.create_image(0, 0, anchor=NW, image=img_tk)

    # Tampilkan hasil teks ke Text widget
    hasil_teks.delete('1.0', END)
    hasil_teks.insert(END, f"Plat Nomor  : {hasil_atas}\n")
    hasil_teks.insert(END, f"Bulan/Tahun : {hasil_bawah}")

# Fungsi untuk menyimpan teks hasil OCR
def simpan_teks():
    teks = hasil_teks.get('1.0', END).strip()
    if teks:
        with open("plat_nomor.txt", "w") as file:
            file.write(teks)
        messagebox.showinfo("Info", "Teks berhasil disimpan!")
    else:
        messagebox.showwarning("Warning", "Tidak ada teks untuk disimpan.")

# ================= GUI SETUP ================= #
root = Tk()
root.title("PROJECT ETS MACHINE VISION")
root.configure(bg="#003366")  # Background Biru Tua

# Menetapkan ukuran window menjadi lebih kecil
root.geometry("800x600")  # Ukuran lebih kecil dibandingkan fullscreen

# Frame utama
main_frame = Frame(root, bg="#003366")
main_frame.pack(pady=20)

# Judul dengan informasi nama
judul = Label(main_frame, text="PROJECT ETS MACHINE VISION", font=("Arial", 16, "bold"), fg="white", bg="#003366")
judul.pack(pady=10)

# Informasi nama di bawah judul
info_nama = Label(main_frame, text="M. Naufal Rashif Harisman_0922040018\nM. Friza Firmansyah_0922040015", font=("Arial", 10), fg="white", bg="#003366")
info_nama.pack(pady=5)

# Label dan tombol pilih gambar
judul = Label(main_frame, text="1. Pilih Gambar", font=("Arial", 12, "bold"), bg="#003366", fg="white")
judul.pack(pady=10)

pilih_btn = Button(main_frame, text="📁 Pilih Gambar", font=("Arial", 12), fg="white", bg="#4e9eff", activebackground="#256dff", padx=10, pady=5, command=proses_gambar)
pilih_btn.pack(pady=10)

# Canvas untuk menampilkan gambar (ukuran lebih kecil)
canvas = Canvas(main_frame, width=250, height=150, bg="white", highlightthickness=1, relief="solid")
canvas.pack(pady=10)

# Area untuk menampilkan hasil OCR
hasil_teks_frame = Frame(root, bg="#003366")
hasil_teks_frame.pack(pady=10)

Label(hasil_teks_frame, text="2. Show Plat Nomor", font=("Arial", 12, "bold"), fg="white", bg="#003366").pack()

# Ukuran text box hasil OCR lebih kecil
hasil_teks = Text(hasil_teks_frame, height=4, width=30, font=("Consolas", 11), bg="#ffffff", fg="#333333", wrap=WORD)
hasil_teks.pack(pady=5)

# Tombol untuk menyimpan hasil OCR
simpan_btn = Button(root, text="3. Simpan Teks", font=("Arial", 12), fg="white", bg="#4e9eff", activebackground="#256dff", padx=10, pady=5, command=simpan_teks)
simpan_btn.pack(pady=20)

root.mainloop()


Using CPU. Note: This module is much faster with a GPU.
C:\Users\mnauf\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
